---
title: Feasibility Analysis Notebook
description: Metropolitan Council VMT Reduction Mode Shift Project
author: Greg Erhardt, Ian Zhang
institute: University of Kentucky
date: today
title-block-banner: true
execute:
    echo: false
    output: false
standalone: true
toc: true
format:
    html:
        embed-resources: true
---

# VMT Reduction Feasibility Analysis

The purpose of this script is to define a set of rules that classify which trips in the TBI could feasibly swith to another mode. The result of this analysis will be a set of binary flags attached to each TBI record where the mode is car:

    feasible_walk_shift    - it is feasible for the trip to switch to walk
    feasible_bike_shift    - it is feasible for the trip to switch to bike
    feasible_transit_shift - it is feasible for the trip to switch to transit
    feasible_shift         - it is feasible for the trip to switch to any non-car mode

In all cases, the flag starts as 1 (feasible) and we set it to infeasible if it violates one or more of our rules. Even though our interest is in understanding which car trips could switch to another mode, we code this flag for all modes.  This way, we can see how existing trips on that mode violate the feasibility rules.  In general, we aim to follow the "95% rule" such that 95% of trips that currently use that mode meet the feasibility condition.  Therefore, about 5% of trips will violate our own rules.  This is ok, because these trips are the most dedicated users of that mode, and new users will likely be less dedicated.  

The list below shows several of the factors tested in this script:

- Trip distance (e.g., walk distance > X miles)
- Weather (particularly, snowfall)
- Distance made on higher level of traffic stress routes (biking), calculated from the re-routing analysis
- Number of transfers that would be required for a full transit trip from origin to destination
- Distance to access a transit stop needed to embark on a transit trip
- Timing analysis
  - Some parts of a daily activity pattern are considered fixed, like school/work, and a mode shift from car to some slower mode may not be feasible to a person who has to be on time to these fixed patterns. 

Each of factors tested in this script also has an independent flag for whether a given trip satisfies or violates a feasibility condition. It is 1 if a trip satisfies the feasibiltiy condition (this factor did not lead a trip to be infeasible), and it is 0 if a trip violates the feasibility condition (this factor is one of possibly several factors that led a trip to be infeasible).

Below, some parameters can be found. These define the cutoffs for various feasibility indicators discussed above and adjust some of the figures created in this notebook. Some factors don't have paramters, though, such as timing, due to a lack of configurable options for the field. By default, most parameters are set to the 95th percentiles of various fields or the value associated with the 95th percentile.

In [ ]:
#| echo: true

WALK_DISTANCE_CUTOFF = 1.6 # in miles; any trip that would walk more than 1.6 miles would be infeasible; 95th percentile
WALK_DISTANCE_PCT = 0.95 # percentile to look at for walk distance
BIKE_DISTANCE_CUTOFF = 11.6 # in miles; 95th percentile
BIKE_DISTANCE_PCT = 0.95 # percentile to look at for bike distance

MAX_TRANSIT_STOP_DIST = 1.4 # in miles; max distance to reach a transit stop from an origin for feasibility; 95th percentile
MAX_TRANSIT_STOP_DIST_PCT = 0.95
MAX_NUM_TRANSFERS = 3 # 95th percentile
MAX_NUM_TRANSFERS_PCT = 0.95

MAX_SNOW_DEPTH_BIKE = 0 # max snow depth in mm that is allowed for a biking trip
MAX_SNOW_DEPTH_BIKE_PCT = 0.95

MAX_HIGH_LTS_DIST_BIKE = 250 # max distance on lts 3 + 4 allowed for a feasible bike trip in meters
MAX_HIGH_LTS_DIST_BIKE_PCT = 0.95 # NOTE: this isn't based on canonical TBI data; it is determined by the methodology of the rerouting analysis and how it avoids lts 3/4

## Read and merge data

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import geopandas as gpd

In [ ]:
import math
import keyring
import itertools
from ast import literal_eval

In [ ]:
pd.set_option('display.max_rows', 500)
pd.set_option('display.max_columns', 500)
pd.set_option('display.width', 1000)

In [ ]:
# This is used to avoid hard-coding directories. 
# To set the directory, use the command prompt or a notebook you don't check in.  run:
# import keyring
# keyring.set_password("msp", "vmt_reduction_dir", <directory>)

# get base path for data
data_dir = keyring.get_password("msp", "vmt_reduction_dir")

In [ ]:
# read in the data
df = pd.read_csv(data_dir + "/data_processed/tbi_cleaned.csv")

In [ ]:
data_dir

In [ ]:
# read in raw tbi data
wave1_trips = pd.read_csv(data_dir + "/Data/TBI Wave 1 Dataset 20200630/trip.csv")
wave2_trips = pd.read_csv(data_dir + "/Data/Wave 2 Data Deliverable/trip.csv")
raw_trips = pd.concat([wave1_trips, wave2_trips]).set_index("trip_id")
raw_trips

In [ ]:
# reading in weather data
# https://www.ncei.noaa.gov/pub/data/ghcn/daily/
weather = pd.read_csv("extra_data/USW00014922.csv")
weather = weather[["Date", "Measurement", "Value"]]
weather = weather.pivot(index="Date", columns="Measurement", values="Value")
weather

In [ ]:
weather["year"] = weather.index.astype(str).str[0:4]
weather["month"] = weather.index.astype(str).str[4:6]
weather["day"] = weather.index.astype(str).str[6:8]
weather["date"] = weather["year"] + "-" + weather["month"] + "-" + weather["day"]
weather = weather.set_index("date")
weather

In [ ]:
# see https://www.ncei.noaa.gov/pub/data/ghcn/daily/readme.txt for a key
df["temperature"] = (weather.loc[df["travel_date"].values, "TAVG"] / 10).values # average temperature in Celsius
df["precipitation"] = (weather.loc[df["travel_date"].values, "PRCP"] / 10).values # precipition in mm
df["snowfall"] = (weather.loc[df["travel_date"].values, "SNOW"] / 10).values # snowfall in mm
df["temperature_max"] = (weather.loc[df["travel_date"].values, "TMAX"] / 10).values # average temperature in Celsius
df["temperature_min"] = (weather.loc[df["travel_date"].values, "TMIN"] / 10).values # average temperature in Celsius
df["precipitation"] = (weather.loc[df["travel_date"].values, "PRCP"] / 10).values # precipition in mm
df["snow_depth"] = (weather.loc[df["travel_date"].values, "SNWD"] / 10).values # snowfall in mm

In [ ]:
# read path geopkg data
# parquet files take an order of magnitude less time to read
car = pd.read_parquet(data_dir + "/Data_Processed/geodata/car_congestion_nogeom.parquet")
bike = gpd.read_parquet(data_dir + "/Data_Processed/geodata/bike_lts.parquet")
transit = gpd.read_parquet(data_dir + "/Data_Processed/geodata/transit_trips.parquet")
walk = pd.read_parquet(data_dir + "/Data_Processed/geodata/walk_trips_nogeom.parquet")

In [ ]:
# calculate duration from star/tend times
transit["start_time_dt"] = pd.to_datetime(transit["start_time"])
transit["end_time_dt"] = pd.to_datetime(transit["end_time"])

transit["duration"] = (transit["end_time_dt"] - transit["start_time_dt"]).apply(lambda x: x.seconds / 60)

In [ ]:
transit["num_transfers"] = (transit["leg_type"] == "TransitRouter.transfer") # temporary field to calculate number of transfers
transit["non_transit_duration"] = (transit["leg_type"].isin(["TransitRouter.access", "TransitRouter.egress"])) * (transit["duration"]) # non-transit time (access and egress)

In [ ]:
# project to US equidistant projection to calculate lengths of paths (in meters)
# https://spatialreference.org/ref/esri/usa-contiguous-equidistant-conic/
transit = transit.to_crs("ESRI:102005")

In [ ]:
transit["length"] = transit.length
transit["access_length"] = transit["length"]
transit

In [ ]:
# create a gdf for linked trips
agg_fns = {
    "trip_id": "first",
    "leg_index": "count",
    "start_time": "first",
    "end_time": "last",
    "origin_stop_id": "first", # throw away
    "origin_stop_name": "first",
    "destination_stop_id": "first",
    "destination_stop_name": "first",
    "route_id": "first", 
    "route_short_name": "first",
    "route_long_name": "first",
    "route_type": "first", 
    "leg_type": "first", # end throw away
    "start_time_dt": "first",
    "end_time_dt": "last",
    "duration": "sum",
    "length": "sum",
    "access_length": "first",
    "num_transfers": "sum",
    "non_transit_duration": "sum"
}

transit_grouped = transit.dissolve(by="trip_id", aggfunc=agg_fns) # merges geometries in addition to aggregating the rest of the columns

In [ ]:
# rename columns to distinguish by mode
def add_mode_to_column_name(df, mode_name): 
    renames = {}
    for col in df.columns:
        renames[col] = mode_name + '_' + col
    return df.rename(columns = renames)

car = add_mode_to_column_name(car, 'car')
walk = add_mode_to_column_name(walk, 'walk')
bike = add_mode_to_column_name(bike, 'bike')
transit = add_mode_to_column_name(transit_grouped, 'transit')

In [ ]:
def get_car_data(row, field):
    hour = int(row["arrive_time"][0:2])
    sunday = row["travel_dow"] == "Sunday"
    saturday = row["travel_dow"] == "Saturday"
    if sunday:
        query = "sundays"
    elif saturday:
        query = "saturdays_"
    else:
        query = "weekdays_"
    
    if hour >= 0 and hour <= 5:
        query += "0-6"
    elif hour >= 20 and hour <= 23:
        query += "20-24"
    else:
        query += str(hour) + "-" + str(hour + 1)

    if (query, row["trip_id"]) not in car.index:
        print(row["trip_id"])
        return np.nan

    return car.loc[(query, row["trip_id"])][field]

In [ ]:
# missing one trip for some reason
df["car_duration_seconds"] = df.apply(lambda x: get_car_data(x, "car_duration_seconds"), axis=1)
df["car_distance_meters"] = df.apply(lambda x: get_car_data(x, "car_distance_meters"), axis=1)

In [ ]:
df = df.merge(walk, left_on="trip_id", right_on="walk_trip_id", how="left")
df = df.merge(bike.drop("bike_geometry", axis=1), left_on="trip_id", right_on="bike_trip_id", how="left")
df = df.merge(transit.drop("transit_geometry", axis=1), left_on="trip_id", right_on="transit_trip_id", how="left")
df

To start with, everyone is allowed to do everything feasibly.

In [ ]:
#| echo: true
# everything is feasible initially
df['feasible_walk_shift'] = True
df['feasible_bike_shift'] = True
df['feasible_transit_shift'] = True
df['feasible_shift'] = True

For the analysis done in this notebook, only car, bike/scooter, walk, and transit trips were considered, with school bus and rideshare trips neglected. This is because the latter two trips are very distinct from the former 4, due to their more discretionary/limited nature. 

In [ ]:
#| echo: true
# omit schools bus and taxi/ridehail/carshare -- very different from other modes
valid_modes = [("Car","distance"), 
               ("Bike/Scooter","distance"), 
               ("Walk","distance"), 
               ("Transit","distance")]

In [ ]:
def plot_density(column: pd.Series, percentile=0.95, discrete=False, bins=100, size=(12, 6)):
    fig, ax = plt.subplots(figsize=size)
    sns.histplot(column, ax=ax, discrete=discrete, bins=bins, kde=True, stat="density")
    val = column.quantile(q=percentile)
    plt.axvline(x=val, color="red")
    return fig, ax

In [ ]:
def plot_mode_density(df: pd.DataFrame, modes=valid_modes, percentile=0.95, size=(12, 6), bins=300, function=lambda x: x):
    palette = itertools.cycle(sns.color_palette()) # cycle through colors to make sure each mode gets a unique one
    fig, ax = plt.subplots(figsize=size)
    for m in modes: # cycle through all modes
        mode = m[0]
        column = m[1]
        label = mode + ' ' + column
        
        c = next(palette) # get color to use
        group = df[df["mode"] == mode] # filter out the current mode
        sns.histplot(function(group[column]), ax=ax, stat="density", kde=True, label=label, color=c, bins=bins) # plot hist plot with kde overlayed in the color
        val = function(group[column]).quantile(q=percentile) # calculate the value of the given percentile (default 0.95)
        plt.axvline(x=val, color=c) # plot line representing that value on the plot        
        plt.legend()
    return fig, ax

In [ ]:
def show_summaries(df: pd.DataFrame, modes=valid_modes, percentile=[0.95]): # show normal summaries for each mode side by side
    res = []
    labels = []
    if type(percentile) != type([]):
        percentile = [percentile]
    p = [0.25, 0.5, 0.75]
    p += percentile
    p = list(set(p))
    for m in modes:        
        mode = m[0]
        column = m[1]
        labels.append(mode + ' ' + column)
        
        group = df[df["mode"] == mode]
        res.append(group[column].describe(percentiles=p))
    x = pd.concat(res, axis=1)
    x.columns = labels
    return x

## 1. Walk distance > X miles

Walk trips are generally short.  Here we consider the trip length distribution of walking trips, and set a maximum allowable distance to be considered feasible to walk. The parameter for this can be seen near the beginning of the document, but the default is the 95th percentile of the distances of the observed walking trips.

In this figure, the distribution for observed walking trip distances and walking trip distances calculated from the rerouting analysis can be seen. The two lines represent the percentile cutoff specified as a parameter. 

It is clear from comparing these distributions that the distances gathered from the TBI data are relatively similar to the distances gathered from the re-routing analysis, indicating that the re-routing was successful. We can thus reasonably use the re-routing distances as an effective way to approximate the distances non-walking trips would've gone, if they were to switch to walking, which is invaluable for analyzing mode shift potentials. 

In [ ]:
#| output: true

# check how the calculated walk distance compares to observed for walk trips
df['walk_distance_miles'] = df['walk_distance_meters'] / 1609.34
fig, ax = plot_mode_density(df, [('Walk','distance'), ('Walk','walk_distance_miles')], percentile=WALK_DISTANCE_PCT) 
ax.set_xlim(left=0, right=10) 
plt.title("Observed versus rerouted distance disttributions for canonical walk trips")
# roughly similar -- rerouting is effective

In this figure, the distributions for walking distances from the rerouting analysis for observed walk trips can be seen, alongside the distributions for calculated re-routing walking distances for observed car trips. It is readily seen that the car trip walking distances are far more right-skewed than the walking trip walking distances, which is to be expected, as car trips are more likely to be used for long distance trips. 

Of note are the two lines present in the graph, which represent the percentiles of the data specified as a parameter. In particular, any part of the orange distribution to the left of the blue line is within the threshold for feasibility (for the given percentile cutoff) and thus constitutes a feasible mode shift. 

In [ ]:
#| output: true

# compare the the distances car drivers would cover if they were to walk with the distances walked by those who chose to walk already
# the area in orange to the left of the blue line represents those who could feasibly switch w.r.t the distance  (95th percentile)
fig, ax = plot_mode_density(df, [('Walk','walk_distance_miles'), ('Car','walk_distance_miles')], percentile=WALK_DISTANCE_PCT) 
ax.set_xlim(left=0, right=25)
plt.title("Rerouted walk distance distribution for canonical walking and car trips")

Here, the summary statistics for walk trip walk distances and car trip car distances can be seen. This largely reflects the distribution seen above, although the percentile given as a parameter is quantified here. 

In [ ]:
#| output: true

show_summaries(df, modes=[('Walk','walk_distance_miles'), ('Car','walk_distance_miles')], percentile=WALK_DISTANCE_PCT)

By applying the cutoff specified as a parameter earlier (95 percentile at default), we are able to restrict the amount of car trips that could feasibly switch to walking. 

In [ ]:
#| output: true

# set the maximum feasible walking distance to WALK_DISTANCE_CUTOFF (default 1.6 miles)
df["within_feasible_walking_dist"] = True

percent_before = len(df[(df['mode']=='Car') & (df['feasible_walk_shift'])]) / len(df[df['mode']=='Car']) * 100
print("Before constraint ", percent_before , " percent of car trips could shift to walk.")

prev_vmt = df[(df["mode"] == "Car") & (df["feasible_walk_shift"])]["vmt"].sum()

df.loc[df['walk_distance_miles'] > WALK_DISTANCE_CUTOFF, 'feasible_walk_shift'] = False
df.loc[df["walk_distance_miles"] > WALK_DISTANCE_CUTOFF, "within_feasible_walking_dist"] = False

percent_after = len(df[(df['mode']=='Car') & (df['feasible_walk_shift'])]) / len(df[df['mode']=='Car']) * 100
print("After constraint ", percent_after , " percent of car trips could shift to walk.")


In [ ]:
#| output: true

reduced_vmt = df[(df["mode"] == "Car") & df["feasible_walk_shift"]]["vmt"].sum()
print("Before this constraint, {0}% of VMT could be mitigated by switches to walking.".format(prev_vmt / df[(df["mode"] == "Car")]["vmt"].sum() * 100))
print("After this constraint, {0}% of VMT could be mitigated by switches to walking.".format(df[(df["mode"] == "Car") & (df["feasible_walk_shift"])]["vmt"].sum() / df[(df["mode"] == "Car")]["vmt"].sum() * 100))

## 2. Bike distance > X miles

Bike trips are generally on the shorter side.  Here we consider the trip length distribution of walking trips, and set a maximum allowable distance to be considered feasible to walk (as before, this can be specified in the parameter section). By default, the maximum allowable distance is the 95th percentile of distances of the observed biking trips. 

Here, similar to what was previously seen in section 1, the distributions of the observed distances versus the rerouted distances for the observed biking trips can be seen. As before, the distributions mirror each other, meaning we can reasonably use the rerouting distances as a proxy for biking distance in general.

In [ ]:
#| output: true

# check how the calculated bike distance compares to observed for bike trips
df['bike_distance_miles'] = df['bike_distance_meters'] / 1609.34
fig, ax = plot_mode_density(df, [('Bike/Scooter','distance'), ('Bike/Scooter','bike_distance_miles')], percentile=BIKE_DISTANCE_PCT) 
ax.set_xlim(left=0, right=25) 
# calculated generally mirrors the observed, although it is a bit more right-tailed
plt.title("Re-routed versus observed distance distribution for canonical bike trips")

In this figure, we can see a comparison between the distributions of biking distance for observed biking trips versus biking distance if observed car trips were to shift to biking. The car distance distribution is more right-skewed, which is reasonable as cars generally go longer distances. The line here represents the specified percentile of each distribution, and all trips of the orange distribution to the left of the blue line can feasibly switch to biking, as their biking distance is reasonable compared to observed biking distances.

In [ ]:
#| output: true

# check how the calculated bike distances for car trips compare to the calculated distances for observed bike trips
# all car trips to the left of the blue line can feasibly switch to biking (95th percentile)
fig, ax = plot_mode_density(df, [('Bike/Scooter','bike_distance_miles'), ('Car','bike_distance_miles')], percentile=BIKE_DISTANCE_PCT) 
ax.set_xlim(left=0, right=25)
plt.title("Re-routed distance distributions for canonical bike and car trips")

Below, we can see summary statistics for biking/car biking distance. This largely reflects what was seen in the above figure, but with more quantifiable values.

In [ ]:
#| output: true

show_summaries(df, modes=[('Bike/Scooter','bike_distance_miles'), ('Car','bike_distance_miles')], percentile=BIKE_DISTANCE_PCT)

By applying the specified biking distance cutoff (by default, it is the 95th percentile), the percent of car trips that could feasibly switch to biking is reduced. 

In [ ]:
#| output: true

# set the maximum feasible walking distance to BIKE_DISTANCE_CUTOFF
df["within_feasible_biking_dist"] = True

percent_before = len(df[(df['mode']=='Car') & (df['feasible_bike_shift'])]) / len(df[df['mode']=='Car']) * 100
print("Before constraint ", percent_before , " percent of car trips could shift to bike.")

prev_vmt = df[(df["mode"] == "Car") & (df["feasible_bike_shift"])]["vmt"].sum()

df.loc[df['bike_distance_miles'] > BIKE_DISTANCE_CUTOFF, 'feasible_bike_shift'] = False
df.loc[df['bike_distance_miles'] > BIKE_DISTANCE_CUTOFF, 'within_feasible_biking_dist'] = False

percent_after = len(df[(df['mode']=='Car') & (df['feasible_bike_shift'])]) / len(df[df['mode']=='Car']) * 100
print("After constraint ", percent_after , " percent of car trips could shift to bike.")

In [ ]:
#| output: true

reduced_vmt = df[(df["mode"] == "Car") & df["feasible_bike_shift"]]["vmt"].sum()
print("Before this constraint, {0}% of VMT could be mitigated by switches to biking.".format(prev_vmt / df[(df["mode"] == "Car")]["vmt"].sum() * 100))
print("After this constraint, {0}% of VMT could be mitigated by switches to biking.".format(df[(df["mode"] == "Car") & (df["feasible_bike_shift"])]["vmt"].sum() / df[(df["mode"] == "Car")]["vmt"].sum() * 100))

## 3. Transit distance to a transit stop > X miles

Transit trips are a bit more complicated, because they come in several legs.  For example: 

    Leg 1: Walk to Bus Stop
    Leg 2: Ride Bus to Another Stop
    Leg 3: Walk to Destination
    
Here we consider the maximum walking distance to or from a bus stop.  Here we can treat each walking leg on a transit trip as a separate observation.

The cutoff is configurable, as before, and is, by default, set to the 95th percentile.

It should be noted that currently, the transit re-routing already restricts the possibilty of switching to a transit trip, as it only returns a trip if the re-routing conditions are satisifed. This means this condition is already implicitly enforced in the re-routing analysis.

In [ ]:
def get_dist_to_stop(trips, wave): # calculate distance to get to stop to start transit trip
    target = raw_trips.loc[trips[0]] # consider the first trip of the linked trip
    # if the purpose of the trip is to switch modes and the mode type is not to switch modes and the distance is not none, return it
    if target["d_purpose_category"] in [10, 11] and (wave == 1 and target["mode_type"] not in [1, 3, 4] or wave == 2 and target["mode_type"] not in [12, 13, 14]):
        if not np.isnan(target["distance"]):
            return target["distance"]
    # otherwise assume that transit can be reached in 0.1 miles
    return 0.1

In [ ]:
df["est_observed_transit_access_dist"] = 0
df.loc[df["mode"] == "Transit", "est_observed_transit_access_dist"] = df[df["mode"] == "Transit"].apply(lambda x: get_dist_to_stop(literal_eval(x["trip_id"]), x["wave"]), axis=1)

Below is the observed versus estimated (via re-routing) distance to a transit stop. The large disparities between the two is due to the fact that re-routing implicitly applies certain conditions to routes and because the observed distances are estimated, as they aren't explicitly calculatable from the raw TBI data. 

However, we will use the approximated observed estimates as the main reference here for determining the cutoff, as it is less dependent on route optimization assumptions.

In [ ]:
#| output: true

# compare distributions of distance to transit stop
df['access_length_miles'] = df['transit_access_length'] / 1609.34
fig, ax = plot_mode_density(df, [('Transit','est_observed_transit_access_dist'), ('Transit','access_length_miles')], percentile=MAX_TRANSIT_STOP_DIST_PCT) 
ax.set_xlim(left=0, right=5)
# overall seems that the two differ from each other significantly, but both are estimates
# we will use the observed estimate as the feasibility reference since that is less dependent on route optimization assumptions
plt.title("Approximated versus re-routed access distance for canonical transit trips")

Below is a comparison between the distributions of the distance to a transit stop for observed transit versus that for observed car trips (re-routing data was used for both distributions to allow for a fairer comparison). It can be seen that generally, observed transit trips tend to have lower distances to transit stops, which makes sense as observed transit trips likely find transit attractive in some way, and this is one way it could be attractive to them.

The lines indicate the specified percentile of each of the distributions, and all of the orange distribution to the left of the blue line represent car trips that can feasibly switch to transit, when considering the access distance indicator. At the default 95th percentile, it can be seen that all these trips can feasibly shift, as this represents re-routing distances, which already account for this factor when re-routing was then. 

In [ ]:
#| output: true

# check how the calculated access length distances for car trips compare to the calculated access lengths for observed transit trips
# all car trips to the left of the blue line can feasibly switch to transit (95th percentile)
fig, ax = plot_mode_density(df, [('Transit','access_length_miles'), ('Car','access_length_miles'),], percentile=MAX_TRANSIT_STOP_DIST_PCT) 
ax.set_xlim(left=0, right=2)
plt.title("Re-routed access length distributions for canoncical transit and car trips")

Here, the summary statistics for observed transit vs observed car trip distance to a transit stop can be seen, with the former drawing on the estimated TBI data and the latter drawing on the re-routing analysis. The specified percentile of the left column, drawing to some degree on the canoncial TBI data, can be used to determine the feasibility cutoff, which, at the default 95th percentile, is about 1.42. 

In [ ]:
#| output: true
show_summaries(df, modes=[('Transit','est_observed_transit_access_dist'), ('Car','access_length_miles'),], percentile=MAX_TRANSIT_STOP_DIST_PCT)

Applying this feasibiltiy indicator to limit transit mode shifts has no effect for the reasons discussed previously. 

In [ ]:
#| output: true

# set the maximum feasible distance to transit stop (access distance) to MAX_TRANSIT_STOP_DIST
df["within_feasible_dist_from_transit_stop"] = True

percent_before = len(df[(df['mode']=='Car') & (df['feasible_transit_shift'])]) / len(df[df['mode']=='Car']) * 100
print("Before constraint ", percent_before , " percent of car trips could shift to transit.")

prev_vmt = df[(df["mode"] == "Car") & (df["feasible_transit_shift"])]["vmt"].sum()

df.loc[df['access_length_miles'] > MAX_TRANSIT_STOP_DIST, 'feasible_transit_shift'] = False
df.loc[df['access_length_miles'] > MAX_TRANSIT_STOP_DIST, 'within_feasible_dist_from_transit_stop'] = False

percent_after = len(df[(df['mode']=='Car') & (df['feasible_transit_shift'])]) / len(df[df['mode']=='Car']) * 100
print("After constraint ", percent_after , " percent of car trips could shift to transit.")

In [ ]:
#| output: true

reduced_vmt = df[(df["mode"] == "Car") & df["feasible_transit_shift"]]["vmt"].sum()
print("Before this constraint, {0}% of VMT could be mitigated by switches to transit.".format(prev_vmt / df[(df["mode"] == "Car")]["vmt"].sum() * 100))
print("After this constraint, {0}% of VMT could be mitigated by switches to transit.".format(df[(df["mode"] == "Car") & (df["feasible_transit_shift"])]["vmt"].sum() / df[(df["mode"] == "Car")]["vmt"].sum() * 100))

## 4. Number of transit transfers > X transfers

Transfers from one transit medium to another is generally a deterrent and inconvenience to passengers riding transit. Here, we consider the maximum number of transit transfers that would still be feasible for an individual that would potentially be switching to a transit mode from another. By default, this cutoff is the 95th percentile of the observed transit trips, and this would be 3 transfers. 

This indicator suffers from the same issues as the access distance indicator does, as it is already accounted for in the re-routing analysis for transit.

In [ ]:
# this function estimates the number of transfers a transit trip would go on
# the main source of error are inconsistencies with linked trips--e.g., having two trips that should be one linked trip put into two separate linked trips
# another source of error is counting the access trip as implicitly one transfer occasionally
def get_num_transfers(trips, wave):
    res = 0
    for trip in trips:
        # count the number of trips whose destination purposes are to change mode
        if wave == 2 and raw_trips.loc[trip]["d_purpose_category"] == 11 or \
            wave == 1 and raw_trips.loc[trip]["d_purpose_category"] == 10:
            res += 1
    return res

In [ ]:
df["transfers"] = 0
df.loc[df["mode"] == "Transit", "transfers"] = df[df["mode"] == "Transit"].apply(lambda x: get_num_transfers(literal_eval(x["trip_id"]), x["wave"]), axis=1)

Below, the distributions of the number of transfers for observed transit trips can be seen, with the left detailing the estimated number from the TBI data (which should be relatively accurate) and the right detailing the calculated numbers from the re-routing analysis. It can be seen here that the re-routing analysis implicitly restricts the number of transfers of the trips it returns, as the max number of transfers in re-routing is 2, compared to the more varied counts on the left with the estimated TBI data. 

We will use the TBI data for determining the feasibility cutoff, as it is less dependent on the assumptions made in the re-routing optimization.

In [ ]:
#| output: true
# check how the calculated transfer count for car trips compare to the observed transfer count for transit trips
fig, ax = plt.subplots(1, 2, figsize=(10, 5))
sns.countplot(x="transfers", data=df[df["mode"] == "Transit"], ax=ax[0])
sns.countplot(x="transit_num_transfers", data=df[df["mode"] == "Transit"], ax=ax[1])
ax[0].set_title("Approximated transfer counts for canonical transit trips")
ax[1].set_title("Re-routed transfer counts for canonical transit trips")

Below, the distribution of the number of transfers for observed transit trips can be seen on the left and the distribution of the number of transfers for observed car trips can be seen on the right (using the re-routing data). Both of these distributions draw on the re-routing analysis to allow for a fairer comparison. It can overall be seen that car trips, if switched to transit, tend to have more transfers. 

In [ ]:
#| output: true

# compare observed distribution of transfer count to calculated distribution of transfer count 
fig, ax = plt.subplots(1, 2, figsize=(10, 5))
sns.countplot(x="transfers", data=df[df["mode"] == "Transit"], ax=ax[0])
sns.countplot(x="transit_num_transfers", data=df[df["mode"] == "Car"], ax=ax[1])
# it seems the calculated transfer count distribution seems to have a hard cap at 2 and varies less than the observed count
# we will use the observed transfers as a cutoff--max 3 transfers for feasibility
ax[0].set_title("Approximated transfer counts for canonical transit trips")
ax[1].set_title("Re-routed optimized transfer counts for canonical car trips")

Below, the summary statistics for transfers for observed transit and observed car trips can be seen, with the former drawing on the TBI data and the latter drawing on the re-routing analysis. The specified percentile in the transit transfers column can be used as the feasibility cutoff. 

In [ ]:
#| output: true

# 95th percentile of (estimated) observed transfers on transit trips is 3 transfers
show_summaries(df, modes=[('Transit','transfers'), ('Car','transit_num_transfers'),], percentile=MAX_NUM_TRANSFERS_PCT)

As with access distance, since the re-routing already enforces a transfer limit, restricting the number of transfers has no effect.

In [ ]:
#| output: true

# set the maximum feasible transfer number to MAX_NUM_TRANSFERS
df["within_feasible_transfer_num_transit"] = True

percent_before = len(df[(df['mode']=='Car') & (df['feasible_transit_shift'])]) / len(df[df['mode']=='Car']) * 100
print("Before constraint ", percent_before , " percent of car trips could shift to transit.")

prev_vmt = df[(df["mode"] == "Car") & (df["feasible_transit_shift"])]["vmt"].sum()

df.loc[df['transit_num_transfers'] > MAX_NUM_TRANSFERS, 'feasible_transit_shift'] = False
df.loc[df['transit_num_transfers'] > MAX_NUM_TRANSFERS, 'within_feasible_transfer_num_transit'] = False

percent_after = len(df[(df['mode']=='Car') & (df['feasible_transit_shift'])]) / len(df[df['mode']=='Car']) * 100
print("After constraint ", percent_after , " percent of car trips could shift to transit.")

In [ ]:
#| output: true

reduced_vmt = df[(df["mode"] == "Car") & df["feasible_transit_shift"]]["vmt"].sum()
print("Before this constraint, {0}% of VMT could be mitigated by switches to transit.".format(prev_vmt / df[(df["mode"] == "Car")]["vmt"].sum() * 100))
print("After this constraint, {0}% of VMT could be mitigated by switches to transit.".format(df[(df["mode"] == "Car") & (df["feasible_transit_shift"])]["vmt"].sum() / df[(df["mode"] == "Car")]["vmt"].sum() * 100))

## 5. A valid transit route was calculated for a trip

As alluded to earlier, this indicator, which reflects whether the re-rerouting analysis returned a valid transit trip for a given trip, supersedes the previous two sections, covering both access distance/transfer count. Thus, by applying this indicator, we effectively apply the previous two simultaneously, restricting the percentage of car trips that can feasibly shift to transit substantially.

In [ ]:
#| output: true

df["has_feasible_transit_route"] = True

percent_before = len(df[(df['mode']=='Car') & (df['feasible_transit_shift'])]) / len(df[df['mode']=='Car']) * 100
print("Before constraint ", percent_before , " percent of car trips could shift to transit.")

prev_vmt = df[(df["mode"] == "Car") & (df["feasible_transit_shift"])]["vmt"].sum()

df.loc[df["transit_trip_id"].isna(), 'feasible_transit_shift'] = False
df.loc[df["transit_trip_id"].isna(), 'has_feasible_transit_route'] = False

percent_after = len(df[(df['mode']=='Car') & (df['feasible_transit_shift'])]) / len(df[df['mode']=='Car']) * 100
print("After constraint ", percent_after , " percent of car trips could shift to transit.")


In [ ]:
#| output: true

reduced_vmt = df[(df["mode"] == "Car") & df["feasible_transit_shift"]]["vmt"].sum()
print("Before this constraint, {0}% of VMT could be mitigated by switches to transit.".format(prev_vmt / df[(df["mode"] == "Car")]["vmt"].sum() * 100))
print("After this constraint, {0}% of VMT could be mitigated by switches to transit.".format(df[(df["mode"] == "Car") & (df["feasible_transit_shift"])]["vmt"].sum() / df[(df["mode"] == "Car")]["vmt"].sum() * 100))

## 6. Snowdepth > 0 for biking

Snowing can impact the feasibility for various modes significant, but this effect is particularly pronounced for biking. This is due to the snow decreasing the traction a bike can get to manuever and due to the discomfort that would arise from traveling through freezing temperatures at fast speeds, exposed fully to the elements. 

Snow depth is measured in mm and is defined by NOAA (https://www.weather.gov/gsp/snow) as the total depth of snow, ice pellets, or ice on the ground at the time of observation, gauged using a measuring stick. This figure is meant to represent the average depth of snow/ice at ground level at the usual measurement site. (snowfall is measured using a snowboard w.r.t. the previous observation--is accumulation of snow over a day).

Below, the summary statistics for the snow depth (in mm) for observed bike/scooter trips can be seen. It should be noted that at the default 95th percentile parameter, the snow depth for these trips remain at 0. 

In [ ]:
#| output: true

# 95% of bike trips occur with no snow depth
display(show_summaries(df, modes=[("Bike/Scooter", "snow_depth")], percentile=MAX_SNOW_DEPTH_BIKE_PCT))
# 24% of all trips occur in some snow depth
len(df[df["snow_depth"] != 0]) / len(df)

Below is a comparison of snow depth distributions between all the different modes. It is clear here that bike/scooter is an outlier, with the specified percentile much more to the left than the other mode percentiles. This indicates that snow depth is a strong and unique indicator for determining whether a bike trip is feasible. 

In [ ]:
#| output: true

# comparative snow depth density/distributions between all modes
# biking is unique in its lack of snow at the 95th percentile
plot_mode_density(df, modes=[("Bike/Scooter", "snow_depth"), ("Car", "snow_depth"), ("Transit", "snow_depth"), ("Walk", "snow_depth")], percentile=MAX_SNOW_DEPTH_BIKE_PCT)
plt.title("Comparison of snow depth distribution for linked trips segmented by mode")

By restricting the feasibility for a shift to biking during days where there is some amount of snowfall, we decrease the percentage of car trips that can feasibly shift to biking. 

In [ ]:
#| output: true

# require all bike trips to occur in 0 snow depth days
df["feasible_snow_depth_biking"] = True

percent_before = len(df[(df['mode']=='Car') & (df['feasible_bike_shift'])]) / len(df[df['mode']=='Car']) * 100
print("Before constraint ", percent_before , " percent of car trips could shift to biking.")

prev_vmt = df[(df["mode"] == "Car") & (df["feasible_bike_shift"])]["vmt"].sum()

df.loc[df["snow_depth"] > MAX_SNOW_DEPTH_BIKE, 'feasible_bike_shift'] = False
df.loc[df["snow_depth"] > MAX_SNOW_DEPTH_BIKE, 'feasible_snow_depth_biking'] = False

percent_after = len(df[(df['mode']=='Car') & (df['feasible_bike_shift'])]) / len(df[df['mode']=='Car']) * 100
print("After constraint ", percent_after , " percent of car trips could shift to biking.")


In [ ]:
#| output: true

reduced_vmt = df[(df["mode"] == "Car") & df["feasible_bike_shift"]]["vmt"].sum()
print("Before this constraint, {0}% of VMT could be mitigated by switches to biking.".format(prev_vmt / df[(df["mode"] == "Car")]["vmt"].sum() * 100))
print("After this constraint, {0}% of VMT could be mitigated by switches to biking.".format(df[(df["mode"] == "Car") & (df["feasible_bike_shift"])]["vmt"].sum() / df[(df["mode"] == "Car")]["vmt"].sum() * 100))

## 7. Distance on LTS 3 and 4 > X meters

The level of traffic stress (lts) quantifies how stressful/difficult it is to bike in a location, ranging from places with dedicated bike lanes at lts 1 to main streets without any developed biking infrasturcture at lts 4. LTS 1 streets are considered safe and comfortable by almost all riders, LTS 2 for most adults, whereas LTS 3 and 4 are more stressful.

The distance through high traffic stress locations an individual would have to bike through to reach a destination is thus likely strong indicator for the feasibility that a trip could switch to biking. Here, we consider the percent distance traveled during a trip on the higher lts categories, 3 and 4. 

We consider the percentage in particular because car trips are naturally longer than biking trips and distances are already accounted for, so percentages even things out for consideration. 

In [ ]:
df["high_stress_biking_distance"] = df["bike_distance_meters_3"] + df["bike_distance_meters_4"]
df["high_stress_biking_pct"] = df["high_stress_biking_distance"] / df["bike_distance_meters"]

As the TBI data does not detail LTS share, we skip directly to the comparison of the distribution of high stress biking distance between observed bike/car trips. it is very clear here from this figure that observed bike trips tend to be far more left-heavy for high stress distance than car trips (if they were to switch to biking), which makes sense as the reason these bike trips are observed is because they are feasible. The lines represent the specified percentiles for each of the two distributions, and any part of the orange distribution to the left of the blue line can feasibly switch to biking, under the high stress biking distance feasibility indicator. 

In [ ]:
#| output: true

# comparing distribution of amount of high stress biking distance existing biking trips have to go through to that potentially switching car trips would have to travel through
# anything to the left of the blue line is within the 95th percentile of high stress biking distance for existing bikers and is thus an upper bound
fig, ax = plot_mode_density(df, modes=[("Bike/Scooter", "high_stress_biking_distance"), ("Car", "high_stress_biking_distance")], percentile=MAX_HIGH_LTS_DIST_BIKE_PCT)
ax.set_xlim(left=0, right=500)
# we will use 250m as a lower cutoff

Here, we can see the summary statistics for high stress biking distance between observed biking/observed car trips, if the latter were to switch to biking. This largely reflects what was seen in the distribution comparison figure. 

In [ ]:
#| output: true

show_summaries(df, modes=[("Bike/Scooter", "high_stress_biking_distance"), ("Car", "high_stress_biking_distance")], percentile=MAX_HIGH_LTS_DIST_BIKE_PCT)

By applying the high stress feasibility restriction, we reduce the percentage of car trips that could feasibly switch modes to biking. 

In [ ]:
#| output: true

# require all bike trips to occur via 250 meters or less on lts 3/4 roads
df["feasible_high_lts_biking"] = True

percent_before = len(df[(df['mode']=='Car') & (df['feasible_bike_shift'])]) / len(df[df['mode']=='Car']) * 100
print("Before constraint ", percent_before , " percent of car trips could shift to biking.")

prev_vmt = df[(df["mode"] == "Car") & (df["feasible_bike_shift"])]["vmt"].sum()

df.loc[df["high_stress_biking_distance"] > MAX_HIGH_LTS_DIST_BIKE, 'feasible_bike_shift'] = False
df.loc[df["high_stress_biking_distance"] > MAX_HIGH_LTS_DIST_BIKE, 'feasible_high_lts_biking'] = False

percent_after = len(df[(df['mode']=='Car') & (df['feasible_bike_shift'])]) / len(df[df['mode']=='Car']) * 100
print("After constraint ", percent_after , " percent of car trips could shift to biking.")


In [ ]:
#| output: true

reduced_vmt = df[(df["mode"] == "Car") & df["feasible_bike_shift"]]["vmt"].sum()
print("Before this constraint, {0}% of VMT could be mitigated by switches to biking.".format(prev_vmt / df[(df["mode"] == "Car")]["vmt"].sum() * 100))
print("After this constraint, {0}% of VMT could be mitigated by switches to biking.".format(df[(df["mode"] == "Car") & (df["feasible_bike_shift"])]["vmt"].sum() / df[(df["mode"] == "Car")]["vmt"].sum() * 100))

## 8. Timing

For some people, it is infeasible to switch from a car mode to an alternative mode due to timing constraints; with the longer durations of alternative modes, a person may not have the ability to do everything they need to in a given day.

Here, trips are feasible with respect to timing if it is possible to fit in all the fixed trips of a day (which have purposes related to work or school), even if the more discretionary, non-fixed trips of a day (e.g., shopping, social visits) may have to be rescheduled or cancelled. 

As mentioned prior, the trips that are considered fixed have purposes relating to work or school. 

In [ ]:
#| echo: true

fixed_purposes = ["Work", "Work-related", "Escort", "School", "School-related"]

In [ ]:
def convert_to_minutes(str):
    hours, minutes, seconds = [int(x) for x in str.split(":")]
    return hours * 60 + minutes + seconds / 60

def evaluate_timing(df, alt_mode_times):
    return df.groupby(["wave", "person_id", "travel_date"]).apply(lambda x: evaluate_feasible_timing(x, alt_mode_times))

def evaluate_feasible_timing(chunk, alt_mode_times: str):
    # if there is only an inbound and outbound trip, don't need to worry about timing
    if len(chunk) == 2:
        return True
    leg_starts = chunk["depart_time"].apply(lambda x: convert_to_minutes(x)).values
    ref = leg_starts[0]
    # start times, starting by 0 and accounting for midnight wraparound with the mod function, for each leg of the complete tour
    leg_starts = [(x - ref) % 1440 for x in chunk["depart_time"].apply(lambda x: convert_to_minutes(x)).values]
    leg_durations = chunk["duration"].values
    # calculate end times relative to the start times using the duration category
    leg_ends = [(x + y) for (x, y) in zip(leg_starts, leg_durations)]
    # these are arrays indicating whether each leg of the complete tour is a fixed arrival/departure
    fixed_arrivals = chunk["d_purpose_category"].isin(fixed_purposes).values
    fixed_departures = chunk["o_purpose_category"].isin(fixed_purposes).values
    
    # sanity check; if the atlernative time for any of the legs can't be found, return False (means can't route it feasibly, usually for transit)
    # do this as preprocessing
    # if ~(chunk["trip_id"].isin(alt_mode_times.index).any()):
    #     return False
    # alternative durations for each of the legs
    alt_durations = chunk[alt_mode_times].values # make it a col in the dataframe to simplify things
    # if any invalid durations, means routing wasn't possible (generally only for transit)
    # return true since this isn't due to timing issues
    if -1 in alt_durations:
        return True
    
    # return true if there aren't any fixed things to work around
    # also return true if there is only one fixed thing--all trips of these kind can be boiled down to traveling to the fixed thing and traveling back if all non-fixed, discretionary trips are omitted; which can be scheduled around feasibly
    if fixed_arrivals.sum() <= 1:
        return True
    
    # keeps track of a previous fixed arrival trip to compare against a current one (see whether they overlap)
    prev_fixed_arrival = -1
    for i in range(len(chunk)):
        if fixed_arrivals[i]: # if current trip is fixed arrival
            if prev_fixed_arrival != -1: # if there exists some previous fixed arrival trip
                if leg_ends[i] - alt_durations[i] < leg_ends[prev_fixed_arrival]: # if there is an overlap between this trip and the pregvious fixed arrival trip, not feasible
                    return False
            prev_fixed_arrival = i # update previous fixed arrival trip
            
    # basically the same as the above, except work backwards since we want to consider whether the next trip would overlap
    next_fixed_departure = -1
    for i in range(len(chunk)-1, -1, -1):
        if fixed_departures[i]:
            if next_fixed_departure != -1:
                if leg_starts[i] + alt_durations[i] > leg_starts[next_fixed_departure]:
                    return False
                next_fixed_departure = i

    # feasible if nothing is weird
    return True

### 8a. Walking timing analysis

In [ ]:
# walking timing analysis
df["walk_duration"] = df["walk_duration_seconds"] / 60
feasible_walking = evaluate_timing(df, "walk_duration")

In [ ]:
feasible_walking.sum() / len(feasible_walking)

In [ ]:
feasible_walking = feasible_walking.reset_index().rename(columns={0: "feasible_walking"})
df = df.merge(feasible_walking, on=["wave", "person_id", "travel_date"], how="left")

In [ ]:
df["feasible_walking"].sum() / len(df["feasible_walking"])

By applying this timing feasibility restriction to walking trips, we reduce the percentage of car trips that could feasibly switch to walking.

In [ ]:
#| output: true

df["feasible_timing_walking"] = True

percent_before = len(df[(df['mode']=='Car') & (df['feasible_walk_shift'])]) / len(df[df['mode']=='Car']) * 100
print("Before constraint ", percent_before , " percent of car trips could shift to walking.")

prev_vmt = df[(df["mode"] == "Car") & (df["feasible_walk_shift"])]["vmt"].sum()

df.loc[~df["feasible_walking"], 'feasible_walk_shift'] = False
df.loc[~df["feasible_walking"], 'feasible_timing_walking'] = False

percent_after = len(df[(df['mode']=='Car') & (df['feasible_walk_shift'])]) / len(df[df['mode']=='Car']) * 100
print("After constraint ", percent_after , " percent of car trips could shift to walking.")


In [ ]:
#| output: true

reduced_vmt = df[(df["mode"] == "Car") & df["feasible_walk_shift"]]["vmt"].sum()
print("Before this constraint, {0}% of VMT could be mitigated by switches to walking.".format(prev_vmt / df[(df["mode"] == "Car")]["vmt"].sum() * 100))
print("After this constraint, {0}% of VMT could be mitigated by switches to walking.".format(df[(df["mode"] == "Car") & (df["feasible_walk_shift"])]["vmt"].sum() / df[(df["mode"] == "Car")]["vmt"].sum() * 100))

### 8b. Biking timing analysis

In [ ]:
# walking timing analysis
df["bike_duration"] = df["bike_weight"] / 60
feasible_biking = evaluate_timing(df, "bike_duration")

In [ ]:
feasible_biking.sum() / len(feasible_biking)

In [ ]:
feasible_biking = feasible_biking.reset_index().rename(columns={0: "feasible_biking"})
df = df.merge(feasible_biking, on=["wave", "person_id", "travel_date"], how="left")

In [ ]:
df["feasible_biking"].sum() / len(df["feasible_biking"])

By applying this timing feasibility restriction to biking trips, we reduce the percentage of car trips that could feasibly switch to biking.

In [ ]:
#| output: true

df["feasible_timing_biking"] = True

percent_before = len(df[(df['mode']=='Car') & (df['feasible_bike_shift'])]) / len(df[df['mode']=='Car']) * 100
print("Before constraint ", percent_before , " percent of car trips could shift to biking.")

prev_vmt = df[(df["mode"] == "Car") & (df["feasible_bike_shift"])]["vmt"].sum()

df.loc[~df["feasible_biking"], 'feasible_bike_shift'] = False
df.loc[~df["feasible_biking"], 'feasible_timing_biking'] = False

percent_after = len(df[(df['mode']=='Car') & (df['feasible_bike_shift'])]) / len(df[df['mode']=='Car']) * 100
print("After constraint ", percent_after , " percent of car trips could shift to biking.")


In [ ]:
#| output: true

reduced_vmt = df[(df["mode"] == "Car") & df["feasible_bike_shift"]]["vmt"].sum()
print("Before this constraint, {0}% of VMT could be mitigated by switches to biking.".format(prev_vmt / df[(df["mode"] == "Car")]["vmt"].sum() * 100))
print("After this constraint, {0}% of VMT could be mitigated by switches to biking.".format(df[(df["mode"] == "Car") & (df["feasible_bike_shift"])]["vmt"].sum() / df[(df["mode"] == "Car")]["vmt"].sum() * 100))

### 8c. Transit timing analysis

In [ ]:
df["transit_duration"]

In [ ]:
# walking timing analysis
df["transit_duration"] = df["transit_duration"].fillna(-1)
feasible_transit = evaluate_timing(df, "transit_duration")

In [ ]:
feasible_transit.sum() / len(feasible_transit)

In [ ]:
feasible_transit = feasible_transit.reset_index().rename(columns={0: "feasible_transit"})
df = df.merge(feasible_transit, on=["wave", "person_id", "travel_date"], how="left")

In [ ]:
df["feasible_transit"].sum() / len(df["feasible_transit"])

By applying this timing feasibility restriction to transit trips, we reduce the percentage of car trips that could feasibly switch to transit.

In [ ]:
#| output: true

df["feasible_timing_transit"] = True

percent_before = len(df[(df['mode']=='Car') & (df['feasible_transit_shift'])]) / len(df[df['mode']=='Car']) * 100
print("Before constraint ", percent_before , " percent of car trips could shift to transit.")

prev_vmt = df[(df["mode"] == "Car") & (df["feasible_transit_shift"])]["vmt"].sum()

df.loc[~df["feasible_transit"], 'feasible_transit_shift'] = False
df.loc[~df["feasible_transit"], 'feasible_timing_transit'] = False

percent_after = len(df[(df['mode']=='Car') & (df['feasible_transit_shift'])]) / len(df[df['mode']=='Car']) * 100
print("After constraint ", percent_after , " percent of car trips could shift to transit.")


In [ ]:
#| output: true

reduced_vmt = df[(df["mode"] == "Car") & df["feasible_transit_shift"]]["vmt"].sum()
print("Before this constraint, {0}% of VMT could be mitigated by switches to transit.".format(prev_vmt / df[(df["mode"] == "Car")]["vmt"].sum() * 100))
print("After this constraint, {0}% of VMT could be mitigated by switches to transit.".format(df[(df["mode"] == "Car") & (df["feasible_transit_shift"])]["vmt"].sum() / df[(df["mode"] == "Car")]["vmt"].sum() * 100))

# Overall statistics

With these feasibility indicators, we can now determine the trips that can feasibly shift to any mode, whether that is transit, walk, or bike. Using this, we can now report four other metrics: person trips, vehicle trips, cold start number, and vmt before/after applying these feasible mode shifts.

In [ ]:
#| echo: true
df["feasible_shift"] = df["feasible_transit_shift"] | df["feasible_walk_shift"] | df["feasible_bike_shift"]

In [ ]:
#| output: true
print("Percent of car trips that can feasibly switch to an alterantive mode (bike, walk, or transit):")
print(df[df["mode"] == "Car"]["feasible_shift"].sum() / len(df[df["mode"] == "Car"]))
print()

In [ ]:
#| output: true
print("Total number of person trips on car before applying feasible mode shift:")
print(len(df[df["mode"] == "Car"]))
print("Total number of person trips on car after applying feasible mode shift:")
print(len(df[(df["mode"] == "Car") & (~df["feasible_shift"])]))
print()

In [ ]:
#| output: true
print("Total number of vehicle trips on car before applying feasible mode shift:")
print(df[df["mode"] == "Car"]["vehicle_trips"].sum())
print("Total number of vehicle trips on car after applying feasible mode shift:")
print(df[(df["mode"] == "Car") & (~df["feasible_shift"])]["vehicle_trips"].sum())
print()

In [ ]:
#| output: true
print("Total VMT before applying feasible mode shift:")
print(df[df["mode"] == "Car"]["vmt"].sum())
print("Total VMT after applying feasible mode shift:")
print(df[(df["mode"] == "Car") & (~df["feasible_shift"])]["vmt"].sum())
print()

In [ ]:
COLD_START_THRESHOLD = 15

def get_num_cold_starts(chunk):
    leg_starts = chunk["depart_time"].apply(lambda x: convert_to_minutes(x)).values
    ref = leg_starts[0]
    # start times, starting by 0 and accounting for midnight wraparound with the mod function, for each leg of the complete tour
    leg_starts = [(x - ref) % 1440 for x in chunk["depart_time"].apply(lambda x: convert_to_minutes(x)).values]
    leg_durations = chunk["duration"].values
    # calculate end times relative to the start times using the duration category
    leg_ends = [(x + y) for (x, y) in zip(leg_starts, leg_durations)]
    modes = chunk["mode"].values
    
    cold_starts = 0
    prev_end = -1
    # iterate over all trips in a complete tour
    for i in range(len(leg_starts)):
        # if the current leg mode i car
        if modes[i] == "Car":
            # if there wasn't a previous or the difference between the end of the last car trip and the beginning of this car trip is more than 15 minutes, it is a cold start
            if prev_end == -1 or leg_starts[i] - leg_ends[i] > COLD_START_THRESHOLD:
                cold_starts += 1
            # update previous end of car trip
            prev_end = leg_ends[i]
    return cold_starts            

A cold start is defined as a car trip occurring more than 15 minutes after the previous one. These are problematic since cold starts result in more emissions. 

In [ ]:
#| output: true
print("Number of cold starts before applying feasible mode shifts:")
print(df[df["mode"] == "Car"].groupby(["wave", "person_id", "travel_date"]).apply(lambda x: get_num_cold_starts(x)).sum())
print("Number of cold starts after applying feasible mode shifts:")
print(df[(df["mode"] == "Car") & (~df["feasible_shift"])].groupby(["wave", "person_id", "travel_date"]).apply(lambda x: get_num_cold_starts(x)).sum())
print()

Exporting results with flags and all for further analysis and determining probable trips (which are a subset of feasible trips):

In [ ]:
#| echo: true
df.to_csv(data_dir + "/data_processed/feasible_shifts.csv")